# How the denoiser works

This notebook walks through the spectral subtraction in `src/specsub.ts`, one step at a
time, on `testdata/throwitaway.noisy.wav` ("throw it away" spoken in car noise, 8 kHz).

The numbers are not recomputed in Python. `src/trace.ts` runs the real TypeScript code
and saves what happens inside it on every frame; this notebook only loads and plots
those files. So what you see here is exactly what the CLI does.

For every frame of 32 ms (16 ms step) and every frequency bin $k$:

$$
N_k = \operatorname{median}\big(|X_k|^2 \text{ over the last } 0.5\,\text{s}\big),
\qquad
|Y_k|^2 = \max\big(|X_k|^2 - \alpha N_k,\ \beta |X_k|^2\big),
\qquad
G_k = \sqrt{|Y_k|^2 / |X_k|^2}
$$

The output spectrum is $G_k X_k$: the noisy phase is kept, only the magnitude changes.

Requirements: Node.js 23.6 or later on the `PATH`, and numpy, matplotlib and IPython.

In [ ]:
import json
import subprocess
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Audio, display

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
WAV = ROOT / "testdata" / "throwitaway.noisy.wav"
TRACE = ROOT / "out" / "trace"
ALPHAS = [0.25, 1, 2, 4]

plt.rcParams.update({"figure.figsize": (11, 3.6), "figure.dpi": 100, "axes.grid": False})

## 1. Run the TypeScript code and load its internals

One run per value of the over-subtraction factor $\alpha$, so we can compare them at the
end. Each run writes `power.f32`, `noise.f32`, `gain.f32` (frames by bins), the
denoised `clean.wav` and a `meta.json`.

In [ ]:
def run_trace(wav, name, *args):
    # Runs src/trace.ts on `wav` into out/trace/<name>; extra args e.g. "--alpha", "2".
    r = subprocess.run(
        ["node", "src/trace.ts", str(Path(wav).relative_to(ROOT)), str((TRACE / name).relative_to(ROOT)), *map(str, args)],
        cwd=ROOT, capture_output=True, text=True, check=True,
    )
    print(r.stdout.strip())


for a in ALPHAS:
    run_trace(WAV, f"alpha-{a}", "--alpha", a)


def read_wav16(path):
    # 16-bit PCM mono, as written by src/wav.ts (also reads the test file, which is the same format)
    b = Path(path).read_bytes()
    pos = 12
    while pos + 8 <= len(b):
        cid, size = b[pos:pos + 4], int.from_bytes(b[pos + 4:pos + 8], "little")
        if cid == b"fmt ":
            fs = int.from_bytes(b[pos + 12:pos + 16], "little")
        if cid == b"data":
            return fs, np.frombuffer(b[pos + 8:pos + 8 + size], "<i2") / 32768.0
        pos += 8 + size + (size & 1)
    raise ValueError("no data chunk")


def load(name):
    d = TRACE / name
    m = json.loads((d / "meta.json").read_text())
    shape = (m["frames"], m["bins"])
    t = {k: np.fromfile(d / f"{k}.f32", "<f4").reshape(shape).T for k in ("power", "noise", "gain")}  # [bins][frames]
    t["meta"] = m
    t["clean"] = read_wav16(d / "clean.wav")[1]
    return t


runs = {a: load(f"alpha-{a}") for a in ALPHAS}
tr = runs[2]  # the default, alpha = 2
m = tr["meta"]
fs, noisy = read_wav16(WAV)

# Axes: a frame is placed at its centre; bin k is at k * fs / n Hz.
times = (np.arange(m["frames"]) * m["hop"] + m["n"] / 2) / fs
freqs = np.arange(m["bins"]) * fs / m["n"]
warmup = m["histLen"] * m["hop"] / fs  # time before the median window is full

db = lambda p: 10 * np.log10(np.maximum(p, 1e-12))
print({k: m[k] for k in ("fs", "n", "hop", "histLen", "bins", "frames")}, f"warm-up {warmup:.2f} s")

## 2. Listen first

Noisy input, then the output with the default $\alpha = 2$, $\beta = 0.01$.

In [ ]:
print("noisy")
display(Audio(noisy, rate=fs, normalize=False))
print("clean, alpha = 2")
display(Audio(tr["clean"], rate=fs, normalize=False))

In [ ]:
t = np.arange(len(noisy)) / fs
fig, ax = plt.subplots()
ax.plot(t, noisy, lw=0.6, label="noisy", color="0.6")
ax.plot(t, tr["clean"], lw=0.6, label="clean (alpha = 2)", color="C0")
ax.set(xlabel="time (s)", ylabel="amplitude", title="Waveform", xlim=(0, t[-1]))
ax.legend(loc="upper right")
plt.show()

## 3. The noisy spectrogram $|X|^2$

The car noise is strongest between about 250 Hz and 1 kHz and is roughly steady over
time (almost nothing is left below 250 Hz: the recording was high-pass filtered). The
speech, around 1.0 to 1.7 s, shows up as harmonics (stacked horizontal stripes) and
formant sweeps standing out from the noise.

In [ ]:
def spec(ax, z, title, vmin=None, vmax=None, cmap="magma", label="dB"):
    im = ax.pcolormesh(times, freqs, z, shading="nearest", cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set(xlabel="time (s)", ylabel="frequency (Hz)", title=title)
    plt.colorbar(im, ax=ax, label=label)


P_db = db(tr["power"])
vmax = np.percentile(P_db, 99.5)
vmin = vmax - 70
fig, ax = plt.subplots()
spec(ax, P_db, "Noisy power |X|^2", vmin, vmax)
plt.show()

## 4. One frequency bin over time: why the median finds the noise

Pick a single bin where the speech is clearly visible and follow it through time. Each
frame adds one value to that bin's 0.5 s window (31 frames) and the oldest leaves.
`src/quantile.ts` keeps the window sorted, so the median is just the middle element
(see `sliding-median.md`).

As long as speech occupies that bin for less than half of the window, the middle
element is a noise-only value: speech peaks pile up at the top of the sorted window
and never reach the middle. That is why no voice activity detector is needed.

The shaded area is the first 0.5 s, while the window is still filling up: there the
median is taken over fewer frames, so the estimate is less reliable.

In [ ]:
# Bin where speech stands out most above the noise estimate, between 150 Hz and 2 kHz.
lo, hi = np.searchsorted(freqs, [150, 2000])
contrast = np.percentile(tr["power"][lo:hi], 95, axis=1) / np.median(tr["noise"][lo:hi], axis=1)
k = lo + int(np.argmax(contrast))
a = m["alpha"]

p, n_, g = tr["power"][k], tr["noise"][k], tr["gain"][k]
fig, ax = plt.subplots()
ax.axvspan(0, warmup, color="0.92", label="window filling up")
ax.plot(times, db(p), color="0.5", lw=1, label="noisy power |X|^2")
ax.plot(times, db(n_), color="C1", lw=2, label="noise estimate N (median)")
ax.plot(times, db(a * n_), color="C1", lw=1, ls="--", label=f"alpha * N (alpha = {a:g})")
ax.plot(times, db(p * g**2), color="C0", lw=1.2, label="after subtraction |Y|^2")
ax.set(xlabel="time (s)", ylabel="dB", xlim=(0, times[-1]),
       title=f"Bin {k} ({freqs[k]:.0f} Hz)")
ax.legend(loc="lower right", fontsize=8, ncol=2)
plt.show()

above = np.mean(p[m["histLen"]:] > n_[m["histLen"]:])
print(f"after warm-up, |X|^2 is above N in {above:.0%} of frames for this bin")

Reading the plot:

- Around 50% of the frames sit above the median, as expected for a median: the noise
  fluctuates around it by 10 dB or more from frame to frame.
- The short burst near 1.1 s barely moves the orange line (the median): it occupies
  far less than half of the window.
- Around 1.3 to 1.7 s the speech fills this bin for a large part of the window, and
  the median rises by about 5 dB. This is the method's weak point: speech that lasts
  long enough in a bin starts to be counted as noise, and the subtraction then eats
  into it.
- Where the grey line is above the dashed line ($\alpha N$), the speech survives the
  subtraction (blue).
- Where it is below, the output drops to the floor $\beta |X|^2$, which is 20 dB under
  the noisy power for $\beta = 0.01$: the blue line runs parallel to the grey one, 20 dB
  lower.

## 5. The noise estimate $N$ for every bin

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 3.6), gridspec_kw={"width_ratios": [2.2, 1]})
spec(axes[0], db(tr["noise"]), "Noise estimate N (running median)", vmin, vmax)
axes[0].axvline(warmup, color="w", ls=":", lw=1)

# Spectrum view: the average noisy spectrum vs the average estimate, after warm-up.
after = times > warmup
axes[1].plot(freqs, db(tr["power"][:, after].mean(axis=1)), color="0.5", label="mean |X|^2")
axes[1].plot(freqs, db(tr["noise"][:, after].mean(axis=1)), color="C1", label="mean N")
axes[1].set(xlabel="frequency (Hz)", ylabel="dB", title="Average spectra after warm-up")
axes[1].legend()
plt.tight_layout()
plt.show()

The estimate is smooth in time and keeps the shape of the car noise. The speech
harmonics are gone; the only trace of speech is a faint brighter patch around 450 Hz
near 1.5 s, the same leak as in the single-bin plot above.

On the right, the average noisy spectrum is 2 to 3 dB above the average estimate at
*every* frequency, not only where there is speech. Most of that gap is statistics,
not speech: in a noise-only bin, $|X|^2$ is roughly exponentially distributed, and the
median of an exponential distribution is $\ln 2 \approx 0.69$ times its mean
($-1.6$ dB). So the median underestimates the mean noise power, which is one reason
why a useful $\alpha$ is above 1 (with $\alpha = 1/\ln 2 \approx 1.44$, $\alpha N$
matches the mean noise power).

## 6. The gain mask $G$

Each bin of each frame gets a gain between $\sqrt{\beta}$ (the floor, here
$-20$ dB) and 1 ($0$ dB). The mask is close to 0 dB on the speech and sits on the
floor almost everywhere else.

In [ ]:
G_db = 20 * np.log10(np.maximum(tr["gain"], 1e-6))
floor_db = 10 * np.log10(m["beta"])

fig, axes = plt.subplots(1, 2, figsize=(13, 3.6), gridspec_kw={"width_ratios": [2.2, 1]})
spec(axes[0], G_db, "Gain G (dB)", floor_db, 0, cmap="viridis")
axes[1].hist(G_db.ravel(), bins=60, color="C2")
axes[1].set(xlabel="gain (dB)", ylabel="bins x frames", title="Gain distribution", yscale="log")
plt.tight_layout()
plt.show()

print(f"{np.mean(G_db <= floor_db + 0.01):.0%} of bins x frames are on the floor ({floor_db:.0f} dB)")

## 7. Before and after

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
spec(axes[0], P_db, "Noisy |X|^2", vmin, vmax)
spec(axes[1], db(tr["power"] * tr["gain"] ** 2), "After subtraction |G X|^2 (alpha = 2)", vmin, vmax)
axes[0].set_xlabel("")
plt.tight_layout()
plt.show()

## 8. The over-subtraction factor $\alpha$

$\alpha$ scales how much of the noise estimate is removed.

- Small $\alpha$ (0.25 was the best for speech recognition in the Eurospeech 2001
  study) leaves much of the noise but hardly touches the speech.
- Around $\alpha = 1$ to 2, most noise goes to the floor, but the noise frames that
  happen to fluctuate above $\alpha N$ survive as isolated short tones scattered in
  time and frequency: the "musical noise" typical of spectral subtraction. Look for
  the bright dots in the spectrograms below and listen for a watery, twittering
  background.
- At $\alpha = 4$ fewer noise peaks get through, but weaker parts of the speech
  (higher formants, onsets and endings) start to be cut as well.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 7), sharex=True, sharey=True)
for ax, a in zip(axes.ravel(), ALPHAS):
    r = runs[a]
    spec(ax, db(r["power"] * r["gain"] ** 2), f"alpha = {a:g}", vmin, vmax)
    floor = np.mean(r["gain"] <= np.sqrt(m["beta"]) + 1e-6)
    ax.text(0.01, 0.95, f"{floor:.0%} on floor", transform=ax.transAxes, color="w", va="top")
for ax in axes[0]:
    ax.set_xlabel("")
for ax in axes[:, 1]:
    ax.set_ylabel("")
plt.tight_layout()
plt.show()

for a in ALPHAS:
    print(f"alpha = {a:g}")
    display(Audio(runs[a]["clean"], rate=fs, normalize=False))

## 9. The spectral floor $\beta$

$\beta$ sets the lowest the output can go in each bin: $|Y|^2 \ge \beta |X|^2$, so the
gain never drops below $\sqrt{\beta}$ ($10 \log_{10} \beta$ dB).

- A very low floor ($\beta = 0.001$, $-30$ dB) makes the background between the
  musical-noise dots almost silent, which makes the dots stand out even more.
- A higher floor ($\beta = 0.1$ or $0.3$, $-10$ or $-5$ dB) keeps a scaled-down copy of
  the original noise under everything. Less noise is removed, but that residual noise
  masks the musical noise and the result sounds more natural.

All runs below use $\alpha = 2$.

In [ ]:
BETAS = [0.001, 0.01, 0.1, 0.3]
for b in BETAS:
    run_trace(WAV, f"beta-{b}", "--alpha", 2, "--beta", b)
bruns = {b: load(f"beta-{b}") for b in BETAS}

fig, axes = plt.subplots(2, 2, figsize=(13, 7), sharex=True, sharey=True)
for ax, b in zip(axes.ravel(), BETAS):
    r = bruns[b]
    spec(ax, db(r["power"] * r["gain"] ** 2), f"beta = {b:g} (floor {10 * np.log10(b):.0f} dB)", vmin, vmax)
for ax in axes[0]:
    ax.set_xlabel("")
for ax in axes[:, 1]:
    ax.set_ylabel("")
plt.tight_layout()
plt.show()

# Noise reduction measured in the speech-free part after warm-up (before 1.0 s).
quiet = (times > warmup) & (times < 1.0)
p_quiet = tr["power"][:, quiet].sum()
for b in BETAS:
    r = bruns[b]
    nr = 10 * np.log10(p_quiet / (r["power"][:, quiet] * r["gain"][:, quiet] ** 2).sum())
    print(f"beta = {b:g}: noise reduced by {nr:.1f} dB between {warmup:.1f} and 1.0 s")
    display(Audio(r["clean"], rate=fs, normalize=False))

## 10. The simplest case: a tone on its own, and the two warm-up options

To see the estimator with nothing else in the way, denoise a pure 440 Hz tone, 2 s long,
starting right at $t = 0$. No speech, no car noise. We use $\alpha = 2$ and $\beta = 0$,
so anything counted as noise is removed completely (with the default $\beta = 0.01$ it
would only drop by 20 dB).

Once the median has caught up, $N = |X|^2$ in the tone's bins, so
$|X|^2 - \alpha N = (1 - \alpha)|X|^2 < 0$ for any $\alpha > 1$ and the gain is exactly 0.
The question is **what happens before the 0.5 s window is full**. There are two ways to
start it (`--warmup`):

| | `grow` (the draft's default) | `zeros` (the 2001 demo, `nlss.cpp`) |
|---|---|---|
| Window at the start | empty; the median uses only the frames seen so far | full of zeros (`specArraySort` is reset to 0) |
| First frame of the tone | median of 1 frame = the tone: removed at once | median of 30 zeros and 1 tone frame = 0: passes |
| Tone silenced after | the first frame | more than half the window is tone, about 0.25 s |

With `grow`, a noise that is already there when processing starts is removed from the
first frame. With `zeros`, nothing is subtracted for the first quarter of a second.
Note that for a sound that starts *after* some silence, both options behave the same:
silent frames are zeros anyway.

In [ ]:
import wave

TONE_HZ, TONE_AMP, DUR = 440.0, 0.5, 2.0
WARMUPS = ["grow", "zeros"]
t = np.arange(int(DUR * fs)) / fs
tone = TONE_AMP * np.sin(2 * np.pi * TONE_HZ * t)

TONE_WAV = ROOT / "out" / "tone.wav"
with wave.open(str(TONE_WAV), "wb") as w:
    w.setnchannels(1)
    w.setsampwidth(2)
    w.setframerate(fs)
    w.writeframes(np.round(tone * 32767).astype("<i2").tobytes())

for wu in WARMUPS:
    run_trace(TONE_WAV, f"tone-{wu}", "--alpha", 2, "--beta", 0, "--warmup", wu)
truns = {wu: load(f"tone-{wu}") for wu in WARMUPS}
tm = truns["grow"]["meta"]
ttimes = (np.arange(tm["frames"]) * tm["hop"] + tm["n"] / 2) / fs
half_window = tm["histLen"] * tm["hop"] / fs / 2

# Waveforms: the input tone, and each output.
fig, axes = plt.subplots(2, 1, figsize=(11, 5), sharex=True, sharey=True)
for ax, wu in zip(axes, WARMUPS):
    ax.plot(t, tone, lw=0.5, color="0.75", label="input: tone")
    ax.plot(t, truns[wu]["clean"], lw=0.5, color="C0", label="output")
    ax.set(ylabel="amplitude", title=f"warmup = {wu}", xlim=(0, DUR))
    ax.legend(loc="upper right")
axes[1].set_xlabel("time (s)")
plt.tight_layout()
plt.show()

print("input")
display(Audio(tone, rate=fs, normalize=False))
for wu in WARMUPS:
    print(f"output, warmup = {wu}")
    display(Audio(truns[wu]["clean"], rate=fs, normalize=False))

In [ ]:
# Follow the bin closest to the tone. Silence is 0, so clip the dB scale for display.
kt = int(round(TONE_HZ * tm["n"] / fs))
p = truns["grow"]["power"][kt]  # the input is the same for both runs
lo_db = db(p.max()) - 80
clip = lambda z: np.maximum(db(z), lo_db)

fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True, sharey=True)
for ax, wu in zip(axes, WARMUPS):
    r = truns[wu]
    ax.axvspan(0, half_window, color="C3", alpha=0.12, label="first half-window")
    ax.plot(ttimes, clip(p), color="0.5", lw=3, label="input power |X|^2")
    ax.plot(ttimes, clip(r["noise"][kt]), color="C1", lw=2, ls="--", label="noise estimate N (median)")
    ax.plot(ttimes, clip(p * r["gain"][kt] ** 2), color="C0", lw=1.5, label="output |Y|^2")
    ax.set(ylabel=f"dB (clipped at {lo_db:.0f})", xlim=(0, DUR),
           title=f"warmup = {wu}: bin {kt} ({freqs[kt]:.0f} Hz)")
    ax.legend(loc="center right", fontsize=8)
axes[1].set_xlabel("time (s)")
plt.tight_layout()
plt.show()

# When are the tone's bins (the tone and its leakage) silenced, and what is left elsewhere?
near = slice(kt - 4, kt + 5)
for wu in WARMUPS:
    r = truns[wu]
    gate = (r["gain"][near] == 0).all(axis=0)
    first = int(np.argmax(gate))
    left = (r["power"] * r["gain"] ** 2)[:, first:].sum() / r["power"][:, first:].sum()
    print(f"warmup = {wu:5s}: bins {kt - 4} to {kt + 4} at gain 0 from frame centre {ttimes[first]:.3f} s, "
          f"stay at 0: {gate[first:].all()}; after that, output / input energy = {10 * np.log10(left):.0f} dB")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.6), sharey=True)
panels = [(truns["grow"]["power"], "Input: tone")] + [
    (truns[wu]["power"] * truns[wu]["gain"] ** 2, f"Output, warmup = {wu}") for wu in WARMUPS
]
for ax, (z, title) in zip(axes, panels):
    im = ax.pcolormesh(ttimes, freqs, clip(z), shading="nearest", cmap="magma", vmin=lo_db, vmax=db(p.max()))
    ax.axvline(half_window, color="w", ls=":", lw=1)
    ax.set(xlabel="time (s)", title=title)
axes[0].set_ylabel("frequency (Hz)")
plt.colorbar(im, ax=axes, label="dB")
plt.show()

Both runs end in near perfect silence: the tone and the window's spectral leakage on
either side of 440 Hz get a gain of exactly 0 and keep it. The only difference is the
start: `grow` removes the tone from the first frame, `zeros` lets it through for about a
quarter of a second, the time it takes for tone frames to outnumber the zeros.

The few specks left in the output, more than 60 dB below the tone, come from bins far
from 440 Hz whose power *fluctuates* from frame to frame, so that now and then a frame
lands above $\alpha N$ and passes:

- **0 to 30 Hz**, which holds most of the leftover energy: the tone's spectral leakage
  reaching 0 Hz overlaps with the leakage of its mirror image at $-440$ Hz. The two
  interfere, and as the tone's phase shifts from one frame to the next, the power in
  these bins swings by about 17 dB.
- **Above 2 kHz**, most of the specks by count but almost no energy: the only thing in
  the input there is the **16-bit quantization noise** of the WAV file.

Both are the musical noise of section 8 again, in miniature. With `zeros`, the bins just
above 0 Hz also leak a little right after the switch, until the zeros have left their
window.

This is also the weak point of section 4 seen from the other side: the median does not
know what speech is, only how long a sound stays in a bin. Anything that stays for more
than half the window (a hum, a whistle, an engine harmonic, or a long sustained vowel)
ends up counted as noise.